## Setup

In [ ]:
import copy
import os
import glob
import ast
import sys

import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import QuadMesh

sys.path.append("/home/akubaney/projects/na_mpnn/evaluation")
from na_eval_utils import (
    read_json_file,
    extract_sequences_from_structure,
    prepare_complex_sequence_data,
    calculate_base_pairs_and_loops_from_secondary_structure,
)

In [ ]:
# Add Arial font for plotting.
font_path = "./ARIAL.TTF"
matplotlib.font_manager.fontManager.addfont(font_path)
plt.rcParams['font.family'] = "Arial"

## Extract and save results csvs

In [ ]:
# Needed for family mapping during extraction.
id_to_family_csv = pd.read_csv(
    "/home/akubaney/projects/na_mpnn/evaluation/family_analysis/id_family_mapping.csv"
).fillna("")
id_to_family = dict(zip(id_to_family_csv["id"], id_to_family_csv["family"]))

In [ ]:
# Extraction helper functions.
def parse_model_name(json_path: str, parent_dir: str) -> str:
    """
    Infer the model name from the JSON's path, e.g.
    parent_dir/model_name/some_subdir/design_json/foo.json → model_name
    """
    rel = os.path.relpath(json_path, parent_dir)
    return rel.split(os.sep)[0]

def get_polymer_group(row: dict) -> str | None:
    """
    Returns one of "DNA", "Protein-DNA", "RNA", "Protein-RNA" or None if it's a 
    hybrid (skipped).
    """
    nac = ast.literal_eval(row["nucleic_acid_chain_cluster_ids_chain_types"])
    pcc = ast.literal_eval(row["protein_chain_cluster_ids_chain_types"])

    has_protein = len(pcc) > 0
    has_dna = "polydeoxyribonucleotide" in nac
    has_rna = "polyribonucleotide" in nac
    has_hybrid = "polydeoxyribonucleotide/polyribonucleotide hybrid" in nac

    if has_protein and has_dna and not has_rna and not has_hybrid:
        return "DNA (protein context)"
    if has_protein and has_rna and not has_dna and not has_hybrid:
        return "RNA (protein context)"
    if has_dna and not has_protein and not has_rna and not has_hybrid:
        return "DNA"
    if has_rna and not has_protein and not has_dna and not has_hybrid:
        return "RNA"
    return None

def get_ppm_group(row: dict) -> str | None:
    """
    Classify the source of the PPMs in the row.
    """
    ppm_paths = ast.literal_eval(row["ppm_paths"])

    has_ppm = len(ppm_paths) > 0
    ppm_from_crystal = row["dataset_name"] == "rcsb_cif_na"
    ppm_from_distillation = (row["dataset_name"] == "rf2na_distillation_cis_bp") or (row["dataset_name"] == "rf2na_distillation_transfac")

    if has_ppm and ppm_from_crystal:
        return "Crystal"
    if has_ppm and ppm_from_distillation:
        return "Distillation"
    if has_ppm and not ppm_from_crystal and not ppm_from_distillation:
        return "Other"
    return None

def get_family(row: dict) -> str:
    """
    Label the DNA-binding family for a given row.
    """
    FAMILY_EXTENDED_TO_SHORT = {
        "DM DNA-binding domain": "DM",
        "Fork head domain": "Forkhead",
        "High mobility group box domain": "HMG-box",
        "Homeodomain": "Homeodomain",
        "MADF domain": "MADF",
        "Methyl-CpG DNA binding": "MBD",
        "Myc-type, basic helix-loop-helix (bHLH) domain": "Myc-type bHLH",
        "NAC domain": "NAC",
        "SANT/Myb domain": "SANT/Myb",
        "THAP-type zinc finger": "zf-THAP",
        "Zinc finger C2H2-type": "zf-C2H2",
        "Zinc finger, nuclear hormone receptor-type": "zf-NHR",
        "Zn(2)Cys(6) fungal-type DNA-binding domain": "zf-C6",
        "p53, DNA-binding domain": "p53",
        "p53, DNA-binding domain;p53, tetramerisation domain": "p53",
        # Missing or multi-family lumped into Other.
        "": "Other",
        "Beta-trefoil DNA-binding domain;RBP-J/Cbf11/Cbf12, DNA binding;RBP-Jkappa, IPT domain": "Other",
        "Homeodomain;T-box transcription factor, DNA-binding domain": "Other",
        "Rap1, DNA-binding domain;SANT/Myb domain": "Other",
    }

    family_extended = id_to_family.get(row["id"], "")
    if family_extended not in FAMILY_EXTENDED_TO_SHORT:
        raise KeyError(
            f"Family {family_extended} is missing from"
            " FAMILY_EXTENDED_TO_SHORT."
        )
    family = FAMILY_EXTENDED_TO_SHORT[family_extended]

    return family

def get_polymer_group_detailed(row: dict) -> dict:
    """
    Gets a more detailed breakdown of the nucleic acid type and protein context
    for a given row, from the structure file.
    """
    # Extract the sequences from the structure file.
    na_sequence_data, protein_sequences = extract_sequences_from_structure(row["structure_path"])

    # Determine the complex type and protein context from the sequence data.
    complex_sequence_data = prepare_complex_sequence_data(
        na_sequence_data=na_sequence_data,
        protein_sequences=protein_sequences,
    )
    if complex_sequence_data["is_single_rna_chain"]:
        na_complex_type = "ssRNA"
    elif complex_sequence_data["is_single_dna_chain"]:
        na_complex_type = "ssDNA"
    else:
        na_complex_type = "multimerNA"
    
    return {
        "na_complex_type": na_complex_type,
        "protein_context": "with protein" if complex_sequence_data["has_protein"] else "without protein",
        "num_na_residues": complex_sequence_data["num_na_residues"],
    }

def get_is_pseudoknot(row: dict) -> bool | None:
    """
    Whether the reference DSSR secondary structure for this row contains a
    pseudoknot, i.e. any two base pairs (i_a, j_a) and (i_b, j_b) where
    i_a < i_b < j_a < j_b (the pairs cross). Returns None if the reference
    JSON has no DSSR annotation.
    """
    reference_path = os.path.join(
        "/home/akubaney/projects/na_mpnn/evaluation/evaluation_outputs/design_test_reference",
        row["id"],
        "reference_json",
        f"{row['id']}.json",
    )
    reference_json = read_json_file(reference_path)

    if "dssr" not in reference_json:
        return None
    
    pairs_indices, _ = calculate_base_pairs_and_loops_from_secondary_structure(
        reference_json["dssr"]["secondary_structure"]
    )
    pairs_indices = sorted(pairs_indices)
    for a in range(len(pairs_indices)):
        i_a, j_a = pairs_indices[a]
        for b in range(a + 1, len(pairs_indices)):
            i_b, j_b = pairs_indices[b]
            if (i_a < i_b) and (i_b < j_a) and (j_a < j_b):
                return True
    return False

In [ ]:
def extract_records(
    df: pd.DataFrame,
    json_paths: list[str],
    parent_dir: str,
    metric_fns: dict,        # map metric_name → function(js) or None
    orig_fn,
    name_fn = None,
    path_column: str = "structure_path",
    group_type: str = None,
    include_family: bool = False,
    include_polymer_group_detailed: bool = False,
    include_is_pseudoknot: bool = False,
) -> pd.DataFrame:
    """
    For each JSON in json_paths:
      - read it, extract orig = orig_fn(js)
      - match it to df[path_column]
      - classify it (polymer/ppm)
      - parse model name
      - pull out metrics
    Return a DataFrame of all records.
    """
    recs = []
    for jp in json_paths:
        js = read_json_file(jp)
        orig = orig_fn(js)
        if orig is None:
            continue

        if name_fn is not None:
            # get the name from the JSON
            name = name_fn(js)
            if name is None:
                continue

        # find the CSV row
        match = df[df[path_column] == orig]
        if match.empty:
            continue

        row = match.iloc[0].to_dict()
        # classify
        if group_type == "polymer":
            group = get_polymer_group(row)
            if group is None:
                continue
        elif group_type == "ppm":
            group = get_ppm_group(row)
            if group is None:
                continue
        else:
            group = None

        model = parse_model_name(jp, parent_dir)

        # build the record
        row_dict = {
            path_column: orig,
            "id": row["id"],
            "Name": name if name_fn is not None else None,
            "Model": model,
            "Group": group,
            "dataset_name": row["dataset_name"],
        }
        if include_family:
            row_dict["family"] = get_family(row)
        if include_polymer_group_detailed:
            row_dict.update(get_polymer_group_detailed(row))
        if include_is_pseudoknot:
            row_dict["is_pseudoknot"] = get_is_pseudoknot(row)

        for mname, mfn in metric_fns.items():
            row_dict[mname] = js[mname] if mfn is None else mfn(js)

        recs.append(row_dict)

    records = pd.DataFrame(recs)
    records = records.sort_values(
        by=["Model", "id", "Name"], kind="stable"
    ).reset_index(drop=True)
    
    return records

In [ ]:
design_valid_df = pd.read_csv("/home/akubaney/projects/na_mpnn/evaluation/evaluation_csvs/design_valid.csv")
specificity_valid_df = pd.read_csv("/home/akubaney/projects/na_mpnn/evaluation/evaluation_csvs/specificity_valid.csv")

design_test_df = pd.read_csv("/home/akubaney/projects/na_mpnn/evaluation/evaluation_csvs/design_test.csv")
specificity_test_df = pd.read_csv("/home/akubaney/projects/na_mpnn/evaluation/evaluation_csvs/specificity_test.csv")

In [ ]:
design_valid_plot_df = extract_records(
    design_valid_df, 
    glob.glob(os.path.join("/home/akubaney/projects/na_mpnn/evaluation/evaluation_outputs/design_valid", '*', '*', "design_json", '*.json')), 
    "/home/akubaney/projects/na_mpnn/evaluation/evaluation_outputs/design_valid", 
    group_type = "polymer",
    metric_fns = {
        "Sequence recovery": lambda json_dict: float(json_dict["tool_reported_sequence_recovery"])
    },
    orig_fn = lambda json_dict: json_dict.get("original_input_structure_path")
)
len(design_valid_plot_df)

In [ ]:
specificity_valid_plot_df = extract_records(
    specificity_valid_df,
    glob.glob(os.path.join("/home/akubaney/projects/na_mpnn/evaluation/evaluation_outputs/specificity_valid_scores", '*', '*', '*.json')), 
    "/home/akubaney/projects/na_mpnn/evaluation/evaluation_outputs/specificity_valid_scores", 
    group_type = "ppm",
    metric_fns = {
        "Mean absolute error": lambda json_dict: float(json_dict["mean_absolute_error_dna"]["mean_absolute_error"]),
        "Cross-entropy": lambda json_dict: float(json_dict["cross_entropy_dna"]["cross_entropy"])
    },
    orig_fn = lambda json_dict: read_json_file(json_dict["subject_path"])["original_input_structure_path"]
)
len(specificity_valid_plot_df)

In [ ]:
specificity_valid_hypersweep_plot_df = extract_records(
    specificity_valid_df,
    glob.glob(os.path.join("/home/akubaney/projects/na_mpnn/evaluation/evaluation_outputs/specificity_valid_hypersweep_scores", '*', '*', '*.json')),
    "/home/akubaney/projects/na_mpnn/evaluation/evaluation_outputs/specificity_valid_hypersweep_scores", 
    group_type = "ppm",
    metric_fns = {
        "Mean absolute error": lambda json_dict: float(json_dict["mean_absolute_error_dna"]["mean_absolute_error"]),
        "Cross-entropy": lambda json_dict: float(json_dict["cross_entropy_dna"]["cross_entropy"])
    },
    orig_fn = lambda json_dict: read_json_file(json_dict["subject_path"])["original_input_structure_path"]
)
len(specificity_valid_hypersweep_plot_df)

In [ ]:
design_test_plot_df = extract_records(
    design_test_df,
    glob.glob(
        os.path.join(
            "/home/akubaney/projects/na_mpnn/evaluation/evaluation_outputs/design_test_scores",
            "*",
            "*",
            "*.json",
        )
    ),
    "/home/akubaney/projects/na_mpnn/evaluation/evaluation_outputs/design_test_scores",
    group_type="polymer",
    include_polymer_group_detailed=True,
    include_is_pseudoknot=True,
    metric_fns={
        "Sequence recovery": lambda score_json: float(
            read_json_file(read_json_file(score_json["subject_path"])["design_input_path"])[
                "tool_reported_sequence_recovery"
            ]
        ),
        "GC content": lambda score_json: score_json["gc_content"],
        "ΔGC content": lambda score_json: score_json["delta_gc_content"],
        "AF3 iPTM": lambda score_json: score_json.get("alphafold3_iptm"),
        "AF3 pLDDT": lambda score_json: score_json.get("alphafold3_plddt"),
        "AF3 pTM": lambda score_json: score_json.get("alphafold3_ptm"),
        "AF3 PAE": lambda score_json: score_json.get("alphafold3_pae"),
        "AF3 C1' RMSD": lambda score_json: score_json.get("alphafold3_c1_prime_rmsd"),
        "AF3 C1' LDDT": lambda score_json: score_json.get("alphafold3_c1_prime_lddt"),
        "AF3 C1' GDDT": lambda score_json: score_json.get("alphafold3_c1_prime_gddt"),
        "AF3 protein-aligned C1' RMSD": lambda score_json: score_json.get("alphafold3_protein_aligned_na_c1_prime_rmsd"),
        "RibonanzaNet OKS": lambda score_json: 100 * (
            score_json["ribonanza_net_openknot_score"]
        ) if score_json.get("ribonanza_net_openknot_score") is not None else None,
        "RibonanzaNet pair F1": lambda score_json: score_json.get("ribonanza_net_f1_score_pairs"),
        "RibonanzaNet loop F1": lambda score_json: score_json.get("ribonanza_net_f1_score_loops"),
        "Mean RibonanzaNet F1": lambda score_json: (
            0.5 * (
                score_json["ribonanza_net_f1_score_pairs"]
                + score_json["ribonanza_net_f1_score_loops"]
            )
            if score_json.get("ribonanza_net_f1_score_pairs") is not None
            and score_json.get("ribonanza_net_f1_score_loops") is not None
            else None
        ),
    },
    orig_fn=lambda score_json: read_json_file(read_json_file(score_json["subject_path"])["design_input_path"]).get(
        "original_input_structure_path"
    ),
    name_fn=lambda score_json: score_json.get("subject_name"),
)


In [ ]:
specificity_test_plot_df = extract_records(
    specificity_test_df, 
    glob.glob(os.path.join("/home/akubaney/projects/na_mpnn/evaluation/evaluation_outputs/specificity_test_scores", '*', '*', '*.json')), 
    "/home/akubaney/projects/na_mpnn/evaluation/evaluation_outputs/specificity_test_scores", 
    group_type = "ppm",
    include_family = True,
    metric_fns = {
        "Mean absolute error": lambda json_dict: float(json_dict["mean_absolute_error_dna"]["mean_absolute_error"]),
        "Cross-entropy": lambda json_dict: float(json_dict["cross_entropy_dna"]["cross_entropy"])
    },
    orig_fn = lambda json_dict: read_json_file(json_dict["subject_path"])["original_input_structure_path"]
)
len(specificity_test_plot_df)

# Remove TRANSFAC.
specificity_test_plot_df = specificity_test_plot_df[
    specificity_test_plot_df["dataset_name"] != "rf2na_distillation_transfac"
].copy()

In [ ]:
# Save the dataframes.
design_valid_plot_df.to_csv("/home/akubaney/projects/na_mpnn/evaluation/evaluation_summaries/design_valid_plot.csv", index=False)
specificity_valid_plot_df.to_csv("/home/akubaney/projects/na_mpnn/evaluation/evaluation_summaries/specificity_valid_plot.csv", index=False)
specificity_valid_hypersweep_plot_df.to_csv("/home/akubaney/projects/na_mpnn/evaluation/evaluation_summaries/specificity_valid_hypersweep_plot.csv", index=False)
design_test_plot_df.to_csv("/home/akubaney/projects/na_mpnn/evaluation/evaluation_summaries/design_test_plot.csv", index=False)
specificity_test_plot_df.to_csv("/home/akubaney/projects/na_mpnn/evaluation/evaluation_summaries/specificity_test_plot.csv", index=False)

In [ ]:
polymer_type_palette = {
    "DNA": "#FF4B4B",
    "DNA (protein context)": "#FF7F7F",
    "RNA": "#4B4BFF",
    "RNA (protein context)": "#7F7FFF",
}

ppm_type_palette = {
    "Crystal": "#E0BBE4",
    "Distillation": "#5D3A9B",
}

design_model_palette = {
    "na_mpnn": "#2ECC71",
    "na_mpnn_no_protein": "#82E0AA",
    "grnade": "#D3D3D3",
    "rhodesign": "#A9A9A9",
    "ridiffusion": "#7A7A7A"
}

specificity_model_palette = {
    "na_mpnn": "#2ECC71",
    "deeppbs": "#D3D3D3",
    "rclamps": "#A9A9A9"
}

model_name_to_label = {
    "na_mpnn": "NA-MPNN",
    "na_mpnn_no_protein": "NA-MPNN (no protein)",
    "grnade": "gRNAde",
    "rhodesign": "RhoDesign",
    "ridiffusion": "RIdiffusion",
    "deeppbs": "DeepPBS",
    "rclamps": "rCLAMPS"
}

data_source_name_to_label = {
    "rf2na_distillation_cis_bp": "CIS-BP",
    "rf2na_distillation_transfac": "TRANSFAC"
}

design_models_without_protein = [
    "na_mpnn",
    "grnade",
    "rhodesign",
    "ridiffusion",
]

design_models_with_protein = [
    "na_mpnn",
    "na_mpnn_no_protein",
    "grnade",
    "rhodesign",
    "ridiffusion",
]

specificity_models = [
    "na_mpnn", 
    "deeppbs", 
    "rclamps"
]

metric_vs_model_step_style = {
    "figsize": (125 / 25.4, 50 / 25.4),
    "title_fontsize": None,
    "axis_title_fontsize": 6,
    "tick_labelsize": 5,
    "legend_title_fontsize": None,
    "legend_fontsize": 5,
    "dpi": 300,
    "hide_top_right_spines": True,
    "spine_linewidth": 0.7,
    "tick_width": 0.7,
    "line_width": 0.5,
    "background_color": "white",
    "show_title": False,
    "show_axis_labels": True,
    "show_xticks": True,
    "show_yticks": True,
    "markersize": 3,
    # Legend saving defaults
    "save_legend_separately": True,
    "legend_save_suffix": "_legend",
    "legend_figsize": (2, 2),
    "show_legend": True,
    "legend_title": None,
    "legend_ncol": 1,
    "legend_handlelength": 1.5,
    "legend_handleheight": 0.3,
    "legend_labelspacing": 0.5,
    "legend_markerscale": None,
    "legend_handletextpad": 0.2,
    "legend_handle_linewidth": 0.5,
    "legend_fill_figure": True,
    "legend_frameon": False,
    "legend_borderpad": None,
    "legend_frame_linewidth": None,
    "legend_mode": "expand",
    # Saving controls
    "save_name": None,
}

metric_heatmap_style = {
    "figsize": (120 / 25.4, 100 / 25.4),
    "title_fontsize": 8,
    "axis_title_fontsize": 6,
    "tick_labelsize": 5,
    "legend_title_fontsize": None,
    "legend_fontsize": 5,
    "annot_fontsize": 5,
    "dpi": 300,
    "annot": True,
    "fmt": ".2f",
    "Background color": "white",
    "show_title": False,
    "show_axis_labels": True,
    "show_xticks": True,
    "show_yticks": True,
    "spine_linewidth": 0.7,
    "tick_width": 0.7,
    "line_width": 0.5,
    "tick_length": None,
    "palette": "viridis",
    "vmin": None,
    "vmax": None,
    "cbar": True,
    "cmap": "viridis",
    "cbar_label": None,
    "cbar_label_fontsize": 6,
    # Legend saving defaults (no-op if no legend)
    "save_legend_separately": True,
    "legend_save_suffix": "_legend",
    "legend_figsize": (2, 2),
    "show_legend": False,
    "legend_title": None,
    "legend_ncol": 1,
    "legend_handlelength": 1.5,
    "legend_handleheight": 0.3,
    "legend_labelspacing": 0.5,
    "legend_handle_linewidth": 0.5,
    "legend_markerscale": None,
    "legend_handletextpad": 0.2,
    "legend_fill_figure": True,
    "legend_frameon": False,
    "legend_borderpad": None,
    "legend_frame_linewidth": None,
    "legend_mode": "expand",
    # Saving controls
    "save_name": None,
}

box_style = {
    "figsize": (73 / 25.4, 40 / 25.4),
    "title_fontsize": None,
    "axis_title_fontsize": 6,
    "tick_labelsize": 5,
    "legend_title_fontsize": None,
    "legend_fontsize": 5,
    "dpi": 300,
    "hide_top_right_spines": True,
    "spine_linewidth": 0.7,
    "tick_width": 0.7,
    "box_linewidth": 0.5,
    "background_color": "white",
    "show_title": False,
    "show_axis_labels": True,
    "show_xticks": True,
    "show_yticks": True,
    # Legend saving defaults
    "save_legend_separately": True,
    "legend_save_suffix": "_legend",
    "legend_figsize": (15 / 25.4, 20 / 25.4),
    "show_legend": True,
    "legend_title": None,
    "legend_ncol": 1,
    "legend_markerscale": None,
    "legend_handletextpad": 0.2,
    "legend_handlelength": 1.5,
    "legend_handleheight": 0.3,
    "legend_labelspacing": 0.5,
    "legend_handle_linewidth": 0.5,
    "legend_fill_figure": True,
    "legend_frameon": False,
    "legend_borderpad": None,
    "legend_frame_linewidth": None,
    "legend_mode": "expand",
    # Box or violin control
    "plot_kind": "violin",
    "assert_complete_group_coverage": True,
    # Box style
    "box_width": 0.6,
    "box_gap": 0.3,
    "whisker_linewidth": 0.5,
    "cap_linewidth": 0.5,
    "flier_markersize": 1,
    "flier_linewidth": 0.5,
    "median_linewidth": 0.5,
    "median_linecolor": None,
    "y_min": None,
    "y_max": None,
    "tick_length": None,
    # Violin style
    "violin_width": 0.9,
    "violin_linewidth": 0.5,
    "violin_inner": "box", # "box" | "quart" | "stick" | "point" | None
    "violin_cut": 0,
    "violin_density_norm": "width", # "area" | "count" | "width"
    "violin_saturation": 0.75,
    "violin_inner_kws": None, # dict of kwargs forwarded to the inner marker
    "violin_gap": 0.2,
    "subplot_wspace": 0.1,
    # Horizontal reference line(s)                                                                                                                                    
    "hline_value": None,            # float or list[float]; None disables
    "hline_color": "black",                                                                                                                                           
    "hline_linewidth": 0.7,
    "hline_linestyle": (0, (2.7, 2.0)), # dotted; or "-", "--", "-.", ":", (0, (3, 1, 1, 1)) etc.                                                                         
    "hline_add_tick": True,
    # Saving controls
    "save_name": None,
}

In [ ]:
def extend_ylim_without_new_ticks(ax: plt.Axes, headroom: float):
    """
    Extend the y-axis limit by a fraction `headroom` without changing tick locations.
    """
    if headroom and headroom > 0:
        orig_ticks = ax.get_yticks()
        ymin, ymax = ax.get_ylim()
        ax.set_ylim(ymin, ymax + (ymax - ymin) * headroom)
        ax.set_yticks(orig_ticks)

def subset_df(df: pd.DataFrame, groups=None, models=None) -> pd.DataFrame:
    """
    Return a copy of df filtered by optional groups and models.
    """
    df = df.copy()
    if groups is not None:
        df = df[df["Group"].isin(groups)]
    if models is not None:
        df = df[df["Model"].isin(models)]
    return df

def _save_legend_from_axes(ax, parent_save_name, style_dict):
    """Save legend from ax as a separate figure and remove it from parent."""
    if not parent_save_name:
        return None

    lg = ax.get_legend()
    if lg is None:
        return None

    handles, labels = ax.get_legend_handles_labels()

    try:
        lg.remove()
    except Exception:
        pass

    dpi = style_dict.get('dpi', 300)
    suffix = style_dict.get('legend_save_suffix', '_legend')
    root, ext = os.path.splitext(parent_save_name)
    if not ext:
        ext = '.svg'
    save_path = f"{root}{suffix}{ext}"

    # Legend appearance/style options
    legend_figsize = style_dict.get('legend_figsize', (6, 2))
    legend_ncol = style_dict.get('legend_ncol', 1)
    legend_title = style_dict.get('legend_title', None)
    legend_title_fs = style_dict.get('legend_title_fontsize', None)
    legend_fs = style_dict.get('legend_fontsize', None)
    legend_loc = style_dict.get('legend_loc', 'center')
    legend_mode = style_dict.get('legend_mode', None)
    legend_frameon = style_dict.get('legend_frameon', True)
    legend_fill_figure = style_dict.get('legend_fill_figure', False)

    # Spacing/handle controls
    legend_handlelength = style_dict.get('legend_handlelength', None)
    legend_handleheight = style_dict.get('legend_handleheight', None)
    legend_labelspacing = style_dict.get('legend_labelspacing', None)
    legend_borderpad = style_dict.get('legend_borderpad', None)
    legend_markerscale = style_dict.get('legend_markerscale', None)
    legend_handletextpad = style_dict.get('legend_handletextpad', None)

    # Line widths
    legend_frame_linewidth = style_dict.get('legend_frame_linewidth', None)
    legend_handle_linewidth = style_dict.get('legend_handle_linewidth', None)

    legend_kw = dict(ncol=legend_ncol, loc=legend_loc)
    # frame and mode handled per-branch
    if legend_handlelength is not None:
        legend_kw['handlelength'] = legend_handlelength
    if legend_handleheight is not None:
        legend_kw['handleheight'] = legend_handleheight
    if legend_labelspacing is not None:
        legend_kw['labelspacing'] = legend_labelspacing
    if legend_borderpad is not None:
        legend_kw['borderpad'] = legend_borderpad
    if legend_mode is not None:
        legend_kw['mode'] = legend_mode
    if legend_title is not None:
        legend_kw['title'] = legend_title
    if legend_markerscale is not None:
        legend_kw['markerscale'] = legend_markerscale
    if legend_handletextpad is not None:
        legend_kw['handletextpad'] = legend_handletextpad

    # Create the legend figure
    fig_leg = plt.figure(figsize=legend_figsize, dpi=dpi, constrained_layout=True)

    if legend_fill_figure:
        # Fill the entire figure area with the legend
        ax_leg = fig_leg.add_axes([0, 0, 1, 1])
        ax_leg.axis('off')
        legend_kw['loc'] = legend_loc or 'center'
        legend_kw['mode'] = legend_mode or 'expand'
        legend_kw['frameon'] = style_dict.get('legend_frameon', False)  # default off when filling
        legend = ax_leg.legend(handles, labels, **legend_kw)
    else:
        ax_leg = fig_leg.add_subplot(111)
        ax_leg.axis('off')
        legend_kw['frameon'] = legend_frameon
        legend = ax_leg.legend(handles, labels, **legend_kw)

    # Apply font sizes
    if legend_title and legend_title_fs:
        legend.get_title().set_fontsize(legend_title_fs)
    if legend_fs:
        for txt in legend.get_texts():
            txt.set_fontsize(legend_fs)

    # Apply frame linewidth
    if legend_frame_linewidth is not None and legend.get_frame() is not None:
        legend.get_frame().set_linewidth(legend_frame_linewidth)

    # Apply handle linewidth to visible legend handles
    if legend_handle_linewidth is not None:
        for h in getattr(legend, 'legend_handles', []) or []:
            if hasattr(h, 'set_linewidth'):
                try:
                    h.set_linewidth(legend_handle_linewidth)
                except Exception:
                    pass

    fig_leg.savefig(save_path, dpi=dpi)
    return fig_leg


def configure_figure(ax: plt.Axes, style: dict | None = None):
    """
    Configure a matplotlib Axes with title, axis labels, tick labels, legend,
    and y-axis limits/steps using a `style` dict.

    Recognized style keys:
      # Display options
      "show_title": bool (default False)
      "title": str
      "x_label": str
      "y_label": str
      "background_color": str
      "show_axis_labels": bool (default True)
      "show_xticks": bool (default True)
      "show_yticks": bool (default True)
      "ymax": float
      "y_min": float
      "y_max": float
      "tick_length": float
      "show_legend": bool (default False)
      "legend_title": str
      "legend_loc": str (default 'upper left')
      "legend_ncol": int (default 1)
      "legend_mode": str | None (e.g. 'expand')
      "legend_frameon": bool (default True)
      "legend_handlelength": float
      "legend_handleheight": float
      "legend_labelspacing": float
      "legend_borderpad": float
      "legend_frame_linewidth": float
      "legend_handle_linewidth": float
      "legend_markerscale": float
      "legend_handletextpad": float
      "legend_headroom": float (default 0.0)

      
      # Font sizes
      "title_fontsize": float
      "legend_title_fontsize": float
      "legend_fontsize": float
      "axis_title_fontsize": float
      "tick_labelsize": float

      # Y-axis limits and ticks
      "ymin": float
      "ytick_range": tuple(float, float)
      "eps": float (default 1e-6)

      # Line / spine controls
      "spine_linewidth": float
      "tick_width": float
      "hide_top_right_spines": bool

      # Legend saving (new)
      "save_legend_separately": bool (default False)
      "legend_save_suffix": str (default '_legend')
      "legend_figsize": tuple (default (6, 2))
      "legend_fill_figure": bool (default False)
      "save_name": str (optional; plot functions will set this)
    """
    style = copy.copy(style) if style else {}

    # High level parameters
    show_title = style.get("show_title", False)
    title = style.get("title", None)
    x_label = style.get("x_label", None)
    y_label = style.get("y_label", None)
    show_legend = style.get("show_legend", False)
    legend_title = style.get("legend_title", None)

    background_color = style.get("background_color", None)
    show_axis_labels = style.get("show_axis_labels", True)
    show_xticks = style.get("show_xticks", True)
    show_yticks = style.get("show_yticks", True)

    # Font sizes
    title_fs = style.get("title_fontsize", None)
    legend_title_fs = style.get("legend_title_fontsize", None)
    legend_fs = style.get("legend_fontsize", None)
    axis_title_fs = style.get("axis_title_fontsize", None)
    tick_fs = style.get("tick_labelsize", None)

    # Layout and axis params
    legend_headroom = style.get("legend_headroom", 0.0)
    ymin = style.get("ymin", style.get("y_min", None))
    ymax = style.get("ymax", style.get("y_max", None))
    ytick_range = style.get("ytick_range", None)
    eps = style.get("eps", 1e-6)

    # Title
    if show_title:
        if title is None:
            raise ValueError("If show_title is True, 'title' must be provided in style.")
        if title_fs is not None:
            ax.set_title(title, fontsize=title_fs)
        else:
            ax.set_title(title)

    # Axis labels
    if show_axis_labels:
        if x_label is not None:
            if axis_title_fs is not None:
                ax.set_xlabel(x_label, fontsize=axis_title_fs)
            else:
                ax.set_xlabel(x_label)
        if y_label is not None:
            if axis_title_fs is not None:
                ax.set_ylabel(y_label, fontsize=axis_title_fs)
            else:
                ax.set_ylabel(y_label)
    else:
        ax.set_xlabel("")
        ax.set_ylabel("")
    
    # Remove x-label and x-axis ticks if no label provided
    if x_label is None:
        ax.set_xlabel("")
    if y_label is None:
        ax.set_ylabel("")

    # Tick labels
    if tick_fs is not None:
        ax.tick_params(axis="both", labelsize=tick_fs)
    
    if not show_xticks:
        ax.set_xticks([])
    if not show_yticks:
        ax.set_yticks([])

    # Legend using artists' labels
    if show_legend:
        loc = style.get("legend_loc", "upper left")
        ncol = style.get("legend_ncol", 1)
        frameon = style.get("legend_frameon", True)
        mode = style.get("legend_mode", None)

        # Spacing/handle controls
        legend_handlelength = style.get('legend_handlelength', None)
        legend_handleheight = style.get('legend_handleheight', None)
        legend_labelspacing = style.get('legend_labelspacing', None)
        legend_borderpad = style.get('legend_borderpad', None)
        legend_markerscale = style.get('legend_markerscale', None)
        legend_handletextpad = style.get('legend_handletextpad', None)

        legend_kwargs = dict(loc=loc, ncol=ncol, frameon=frameon)
        if mode is not None:
            legend_kwargs["mode"] = mode
        if legend_handlelength is not None:
            legend_kwargs['handlelength'] = legend_handlelength
        if legend_handleheight is not None:
            legend_kwargs['handleheight'] = legend_handleheight
        if legend_labelspacing is not None:
            legend_kwargs['labelspacing'] = legend_labelspacing
        if legend_borderpad is not None:
            legend_kwargs['borderpad'] = legend_borderpad
        if legend_markerscale is not None:
            legend_kwargs['markerscale'] = legend_markerscale
        if legend_handletextpad is not None:
            legend_kwargs['handletextpad'] = legend_handletextpad

        if legend_title is None:
            legend = ax.legend(**legend_kwargs)
        else:
            legend = ax.legend(title=legend_title, **legend_kwargs)

        # Apply frame and handle linewidths
        legend_frame_linewidth = style.get('legend_frame_linewidth', None)
        if legend_frame_linewidth is not None and legend.get_frame() is not None:
            legend.get_frame().set_linewidth(legend_frame_linewidth)

        legend_handle_linewidth = style.get('legend_handle_linewidth', None)
        if legend_handle_linewidth is not None:
            for h in getattr(legend, 'legend_handles', []) or []:
                if hasattr(h, 'set_linewidth'):
                    try:
                        h.set_linewidth(legend_handle_linewidth)
                    except Exception:
                        pass

        if legend_title_fs is not None and legend.get_title():
            legend.get_title().set_fontsize(legend_title_fs)
        if legend_fs is not None:
            for text in legend.get_texts():
                text.set_fontsize(legend_fs)
        extend_ylim_without_new_ticks(ax, legend_headroom)

    if background_color is not None:
        ax.set_facecolor(background_color)
        ax.figure.patch.set_facecolor(background_color)
    
    # y-axis limits
    if ymin is not None:
        ax.set_ylim(ymin=ymin)
    if ymax is not None:
        ax.set_ylim(ymax=ymax)

    # y-axis tick range
    if ytick_range is not None:
        yt_min, yt_max = ytick_range
        ticks = ax.get_yticks()
        ticks = ticks[(ticks >= (yt_min - eps)) & (ticks <= (yt_max + eps))]
        ax.set_yticks(ticks)

    # Horizontal reference line(s) + optional tick injection                                                                                                          
    hline_value = style.get("hline_value", None)
    if hline_value is not None and any(s.get_visible() for s in ax.spines.values()):
        hline_values = [hline_value] if isinstance(hline_value, (int, float)) else list(hline_value)
                                                                                                                                                                    
        for v in hline_values:
            ax.axhline(                                                                                                                                               
                v,
                color=style.get("hline_color", "black"),
                linewidth=style.get("hline_linewidth", 0.5),
                linestyle=style.get("hline_linestyle", (0, (1, 1))),                                                                                                  
                zorder=10,
            )                                                                                                                                                         
                
        if style.get("hline_add_tick", True):                                                                                                                         
            ymin_lim, ymax_lim = ax.get_ylim()                                                                                                                        
            ticks = [t for t in ax.get_yticks() if ymin_lim - eps <= t <= ymax_lim + eps]                                                                             
            for v in hline_values:                                                                                                                                    
                if ymin_lim <= v <= ymax_lim and not any(abs(t - v) < eps for t in ticks):
                    ticks.append(v)                                                                                                                                   
            ticks.sort()
            ax.set_yticks(ticks)                                                                                                                                      
            ax.set_yticklabels([f"{t:g}" for t in ticks])
            ax.set_ylim(ymin_lim, ymax_lim)

    # Line and spine controls
    spine_lw = style.get("spine_linewidth", None)
    tick_width = style.get("tick_width", None)
    tick_length = style.get("tick_length", None)
    hide_top_right = style.get("hide_top_right_spines", False)

    # Apply spine linewidth if provided
    if spine_lw is not None:
        for spine in ax.spines.values():
            spine.set_linewidth(spine_lw)

    # Apply tick width if provided
    if tick_width is not None:
        ax.tick_params(width=tick_width)
    if tick_length is not None:
        ax.tick_params(length=tick_length)

    # Optionally hide top/right spines
    if hide_top_right:
        if "top" in ax.spines:
            ax.spines["top"].set_visible(False)
        if "right" in ax.spines:
            ax.spines["right"].set_visible(False)

def plot_aggregate_metric_vs_model_step(
    df,
    metric,
    agg="median",
    groups=None,
    models=None,
    style=None,
):
    """
    Creates a line plot of the aggregate metric vs. model step.
    """
    df = subset_df(df, groups, models)
    style = copy.copy(style) if style else {}
    dpi = style.get("dpi", None)
    figsize = style.get("figsize", (10, 6))
    palette = style.get("palette", None)
    save_name = style.get("save_name", None)

    # aggregate by group and model
    agg_df = (
        df
        .groupby(["Group", "Model"])[metric]
        .agg(agg)
        .reset_index(name=f"{agg}_{metric}")
    )
    agg_df["step"] = (
        agg_df["Model"].str.rsplit("_", n=1)
                        .str[-1].astype(int)
    )
    agg_df = agg_df.sort_values("step")

    # Set folded style parameters
    style["x_label"] = "Batches"
    style["y_label"] = f"{agg.title()} {metric.lower()}"
    style["save_name"] = save_name

    # Necessary for legend handle/text alignment.
    base_fs = matplotlib.rcParams['legend.fontsize']
    if style.get("legend_fontsize", None) is not None:
        matplotlib.rcParams['legend.fontsize'] = style["legend_fontsize"]

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi, constrained_layout=True)
    ax = sns.lineplot(
        data=agg_df,
        x="step",
        y=f"{agg}_{metric}",
        hue="Group",
        hue_order=groups,
        palette=palette,
        marker="o",
        linewidth=style.get("line_width", None),
        ax=ax,
        markersize=style.get("markersize", None)
    )

    configure_figure(ax=ax, style=style)

    # Save legend separately if requested
    if style.get("save_legend_separately", False) and save_name:
        _save_legend_from_axes(ax, save_name, style)

    if save_name:
        fig.savefig(save_name, dpi=dpi)

    plt.show()

    matplotlib.rcParams['legend.fontsize'] = base_fs

def plot_metric_heatmap(
    df,
    metric,
    agg="median",
    groups=None,
    models=None,
    style=None,
):
    """
    Creates a heatmap of the aggregate metric by sample size and temperature.
    """
    df = subset_df(df, groups, models)
    style = copy.copy(style) if style else {}
    dpi = style.get("dpi", None)
    annot = style.get("annot", True)
    fmt = style.get("fmt", ".2f")
    figsize = style.get("figsize", (10, 8))
    palette = style.get("cmap", style.get("palette", "viridis"))
    save_name = style.get("save_name", None)

    agg_df = (
        df
        .groupby("Model")[metric]
        .agg(agg)
        .reset_index(name=f"{agg}_{metric}")
    )
    agg_df["n"] = (
        agg_df["Model"].str.rsplit("__").str[0]
                   .str.rsplit("_").str[-1].astype(int)
    )
    agg_df["t"] = (
        agg_df["Model"].str.rsplit("__").str[1]
                   .str.rsplit("t_").str[-1].str.replace("_", ".").astype(float)
    )
    pivot = agg_df.pivot(index="n", columns="t", values=f"{agg}_{metric}")

    style["x_label"] = "Temperature"
    style["y_label"] = "Sample size"
    style["save_name"] = save_name

    # Necessary for legend handle/text alignment.
    base_fs = matplotlib.rcParams['legend.fontsize']
    if style.get("legend_fontsize", None) is not None:
        matplotlib.rcParams['legend.fontsize'] = style["legend_fontsize"]

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi, constrained_layout=True)
    ax = sns.heatmap(
        pivot, 
        annot=annot, 
        annot_kws = {
            "size": style.get("annot_fontsize", 10)
        },
        fmt=fmt, 
        cmap=palette, 
        ax=ax,
        vmin=style.get("vmin", None),
        vmax=style.get("vmax", None),
    )

    # Colorbar control.
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(
        labelsize = style.get('tick_labelsize', 10),
    )
    if style.get('tick_width') is not None:
        cbar.ax.tick_params(width=style.get('tick_width'))
    if style.get('tick_length') is not None:
        cbar.ax.tick_params(length=style.get('tick_length'))
    if style.get("cbar_label"):
        cbar.set_label(
            style["cbar_label"],
            fontsize=style.get("cbar_label_fontsize", style.get("axis_title_fontsize", None)),
        )

    # Rasterization control; to not show grid lines when vectorized in svg.
    for coll in ax.collections:
        if isinstance(coll, QuadMesh):
            coll.set_linewidth(0)
            coll.set_edgecolor('face')
            coll.set_antialiased(False)
            if style.get('rasterize_heatmap', True):
                coll.set_rasterized(True)

    configure_figure(ax=ax, style=style)
    if style.get("save_legend_separately", False) and save_name:
        _save_legend_from_axes(ax, save_name, style)  # no-op if no legend

    if save_name:
        fig.savefig(save_name, dpi=dpi)

    plt.show()

    matplotlib.rcParams['legend.fontsize'] = base_fs

def assert_complete_group_coverage(
    df: pd.DataFrame,
    group_column: str,
    models: list[str] | None = None,
    key_column: str = "id",
) -> None:
    if df.empty:
        raise ValueError("No rows left after filtering.")

    expected_models = list(models) if models is not None else sorted(df["Model"].unique())
    subset = df[df["Model"].isin(expected_models)]
    counts = (
        subset.groupby([group_column, key_column, "Model"])
        .size()
        .unstack("Model", fill_value=0)
        .reindex(columns=expected_models, fill_value=0)
    )

    # Important to check that the non-mpnn methods have the same counts as
    # the mpnn methods.
    non_mpnn_cols = [m for m in counts.columns if "mpnn" not in m.lower()]
    if non_mpnn_cols:
        has_baseline_presence = (counts[non_mpnn_cols] > 0).any(axis=1)
        counts = counts[has_baseline_presence]

    has_missing_model = (counts == 0).any(axis=1)
    has_unequal_counts = counts.nunique(axis=1) > 1
    bad_rows = counts[has_missing_model | has_unequal_counts]
    if not bad_rows.empty:
        raise ValueError(
            "Selected models do not have matched coverage in the requested grouped plot.\n"
            f"group_column={group_column}\n"
            f"expected_models={expected_models}\n\n"
            f"{bad_rows.head(20).to_string()}"
        )

def plot_box(
    df,
    metric,
    groups=None,
    models=None,
    style=None,
    print_stats=True,
):
    """
    Creates a box plot, either by group or model.
    """
    df = subset_df(df, groups, models)
    style = copy.copy(style) if style else {}
    dpi = style.get("dpi", None)
    figsize = style.get("figsize", (10, 6))
    palette = style.get("palette", None)
    model_name_to_label = style.get("model_name_to_label", None)
    data_source_name_to_label = style.get("data_source_name_to_label", None)
    save_name = style.get("save_name", None)
    layout_groups = style.get("layout_groups", None)
    layout_group_widths = style.get("layout_group_widths", None)

    # Infer plot_by if not provided
    plot_by = style.get("plot_by", None) 
    if plot_by not in {"Group", "Model"}:
        plot_by = "Group" if groups else "Model"
    group_by = style.get("group_by", None)
    if group_by is None:
        group_by = plot_by
    
    # Check that all models have the same coverage of groups.
    if style.get("assert_complete_group_coverage", False):
        assert_complete_group_coverage(
            df[df[metric].notna()].copy(),
            group_column=group_by,
            models=style.get("coverage_models", models),
        )

    if model_name_to_label is not None and "Model" in df.columns:
        df["Model"] = df["Model"].map(model_name_to_label).fillna(df["Model"])
        if palette is not None:
            palette = {model_name_to_label.get(k, k): v for k, v in palette.items()}
        if models is not None:
            models = [model_name_to_label.get(m, m) for m in models]
    
    if data_source_name_to_label is not None and "dataset_name" in df.columns:
        df["dataset_name"] = df["dataset_name"].map(data_source_name_to_label).fillna(df["dataset_name"])

    if group_by:
        if group_by == "Group" and groups:
            df = df.sort_values(
                "Group",
                key=lambda x: x.map({g: i for i, g in enumerate(groups)}),
            )
        else:
            df = df.sort_values(group_by)
    
    # Determine hue order
    if plot_by == "Group":
        hue_order = groups
        if groups:
            df = df.sort_values("Group", key=lambda x: x.map({g: i for i, g in enumerate(hue_order)}))
    else:
        hue_order = models
        if models:
            df = df.sort_values("Model", key=lambda x: x.map({m: i for i, m in enumerate(hue_order)}))

    style["y_label"] = metric
    style["save_name"] = save_name

    # Necessary for legend handle/text alignment.
    base_fs = matplotlib.rcParams['legend.fontsize']
    if style.get("legend_fontsize", None) is not None:
        matplotlib.rcParams['legend.fontsize'] = style["legend_fontsize"]

    # Create the figure.
    plot_kind = style.get("plot_kind", "box")
    if plot_kind == "box":
        # Box line width controls
        box_lw = style.get("box_linewidth", None)
        whisker_lw = style.get("whisker_linewidth", box_lw if box_lw is not None else 1)
        cap_lw = style.get("cap_linewidth", whisker_lw)
        median_lw = style.get("median_linewidth", whisker_lw)
        median_linecolor = style.get("median_linecolor", None)

        # Outlier marker size and edge width
        flier_ms = style.get("flier_markersize", None)
        flier_lw = style.get("flier_linewidth", None)

        boxprops = {"linewidth": box_lw} if box_lw is not None else None
        whiskerprops = {"linewidth": whisker_lw}
        capprops = {"linewidth": cap_lw}
        medianprops = {"linewidth": median_lw}
        if median_linecolor is not None:
            medianprops["color"] = median_linecolor
        flierprops = {}
        if flier_lw is not None:
            flierprops["markeredgewidth"] = flier_lw
    elif plot_kind == "violin":
        violin_inner_kws = style.get("violin_inner_kws", None)
        violin_density_norm = style.get("violin_density_norm", "width")
    else:
        raise ValueError(f"Unknown plot_kind: {plot_kind}")

    # Per-group subplots. Each group is its own axes with a restricted
    # hue_order limited to models actually present for this metric in
    # that group, so boxes/violins center within their slot.
    plot_groups = (
        layout_groups
        if layout_groups is not None
        else (
            groups if (group_by == "Group" and groups) else
            df.sort_values(group_by)[group_by].drop_duplicates().tolist()
        )
    )
    present_df = df[df[metric].notna()]
    models_per_group = {
        g: [
            m for m in hue_order
            if ((present_df[group_by] == g) & (present_df[plot_by] == m)).any()
        ]
        for g in plot_groups
    }
    if layout_groups is not None and layout_group_widths is not None:
        width_ratios = [layout_group_widths.get(g, 1) for g in plot_groups]
    else:
        width_ratios = [max(len(models_per_group[g]), 1) for g in plot_groups]

    fig, axes = plt.subplots(
        1, len(plot_groups),
        figsize=figsize,
        dpi=dpi,
        sharey=True,
        gridspec_kw={"width_ratios": width_ratios, "wspace": style.get("subplot_wspace", 0)},
        constrained_layout=style.get("constrained_layout", True),
    )

    if style.get("constrained_layout", True):
        constrained_w_pad = style.get("constrained_w_pad", None)
        constrained_wspace = style.get("constrained_wspace", None)
        constrained_layout_rect = style.get("constrained_layout_rect", None)
        if constrained_w_pad is not None:
            fig.set_constrained_layout_pads(w_pad=constrained_w_pad)
        if constrained_wspace is not None:
            fig.set_constrained_layout_pads(wspace=constrained_wspace)
        if constrained_layout_rect is not None:
            fig.set_constrained_layout_pads(rect=constrained_layout_rect)

    if len(plot_groups) == 1:
        axes = [axes]

    legend_handles_by_label = {}

    for ax, g in zip(axes, plot_groups):
        group_df = df[df[group_by] == g]
        local_hue_order = models_per_group[g]

        if group_df[metric].notna().any() and local_hue_order:
            if plot_kind == "box":
                sns.boxplot(
                    data=group_df,
                    x=group_by,
                    y=metric,
                    hue=plot_by,
                    hue_order=local_hue_order,
                    order=[g],
                    palette=palette,
                    ax=ax,
                    showcaps=True,
                    showfliers=True,
                    legend=True,
                    width=style.get("box_width", 0.8),
                    gap=style.get("box_gap", 0),
                    boxprops=boxprops,
                    whiskerprops=whiskerprops,
                    capprops=capprops,
                    medianprops=medianprops,
                    fliersize=flier_ms,
                    flierprops=flierprops or None,
                )
            else:  # plot_kind == "violin"
                sns.violinplot(
                    data=group_df,
                    x=group_by,
                    y=metric,
                    hue=plot_by,
                    hue_order=local_hue_order,
                    order=[g],
                    palette=palette,
                    ax=ax,
                    legend=True,
                    width=style.get("violin_width", 0.9),
                    gap=style.get("violin_gap", 0),
                    linewidth=style.get("violin_linewidth", None),
                    inner=style.get("violin_inner", "box"),
                    cut=style.get("violin_cut", 0),
                    density_norm=violin_density_norm,
                    saturation=style.get("violin_saturation", 0.75),
                    **({"inner_kws": violin_inner_kws} if violin_inner_kws is not None else {}),
                )            
        else:
            ax.set_xlim(-0.5, 0.5)
            ax.set_xticks([])
            ax.set_xlabel("")
            ax.set_ylabel("")
            for spine in ax.spines.values():
                spine.set_visible(False)

        # Collect this subplot's legend handles into a fig-wide dedup
        # dict, then drop the per-subplot legend so configure_figure can
        # draw one combined legend on the last axes.
        subplot_legend = ax.get_legend()
        if subplot_legend is not None:
            for handle, text in zip(
                getattr(subplot_legend, "legend_handles", []) or [],
                subplot_legend.get_texts(),
            ):
                legend_handles_by_label.setdefault(text.get_text(), handle)
            subplot_legend.remove()

        for label in ax.get_xticklabels():
            label.set_rotation(style.get("x_tick_rotation", 0))
            label.set_ha(style.get("x_tick_horizontalalignment", "center"))
            x_tick_rotation_mode = style.get("x_tick_rotation_mode", None)
            if x_tick_rotation_mode is not None:
                label.set_rotation_mode(x_tick_rotation_mode)
        
        if not style.get("x_axis_in_layout", True):
            ax.xaxis.set_in_layout(False)

    # Hide internal spines / y-ticks so the panel reads as one axes.
    for i, ax in enumerate(axes):
        if i > 0:
            ax.spines["left"].set_visible(False)
            ax.tick_params(left=False, labelleft=False)
        if i < len(axes) - 1:
            ax.spines["right"].set_visible(False)

    # Re-attach a single deduped legend to the last subplot so
    # configure_figure's legend path (below) can re-style it. Only needed
    # when the caller actually wants a legend.
    if legend_handles_by_label and style.get("show_legend", False):
        axes[0].legend(
            list(legend_handles_by_label.values()),
            list(legend_handles_by_label.keys()),
        )

    # Apply configure_figure per-axes with positional tweaks: y-label
    # only on the first axes, x-label only on the middle axes (acts as
    # the panel-wide x-label), legend only on the last axes. `ax` is
    # rebound to the last axes afterwards so the rest of plot_box
    # (legend save, fig.savefig, stats) continues to work unchanged.
    middle_idx = len(axes) // 2
    for i, ax in enumerate(axes):
        subplot_style = copy.copy(style)
        if i != 0:
            subplot_style["y_label"] = None
            subplot_style["show_legend"] = False
        if i != middle_idx:
            subplot_style["x_label"] = None
        configure_figure(ax=ax, style=subplot_style)
    ax = axes[0]  # `_save_legend_from_axes(ax, ...)` below reads from this.

    if style.get("save_legend_separately", False) and save_name:
        _save_legend_from_axes(ax, save_name, style)

    if save_name:
        fig.savefig(save_name, dpi=dpi)

    if print_stats:
        if group_by != plot_by:
            count_table = (
                df.groupby([plot_by, group_by])[metric]
                .count()
                .unstack(group_by, fill_value=0)
            )
            median_table = (
                df.groupby([plot_by, group_by])[metric]
                .median()
                .unstack(group_by)
            )

            if plot_by != "Model" or not models:
                raise NotImplementedError("Currently only supports plot_by='Model' for stats printing.")
            else:
                stats_row_order = list(models)
            if group_by != "Group" or not groups:
                raise NotImplementedError("Currently only supports group_by='Group' for stats printing.")
            else:
                stats_col_order = list(groups)

            count_table = count_table.reindex(
                index=stats_row_order, columns=stats_col_order
            )
            median_table = median_table.reindex(
                index=stats_row_order, columns=stats_col_order
            )

            print(f"Count ({plot_by} x {group_by}):")
            print(count_table)
            print(f"\nMedian ({plot_by} x {group_by}):")
            print(median_table)

            overall_median = (
                df[df[metric].notna()]
                .groupby(plot_by)[metric]
                .median()
                .reindex(stats_row_order)
                .rename("overall_median")
            )

            print(f"\nOverall median ({plot_by}, pooled across all {group_by}):")
            print(overall_median)

    plt.show()

    matplotlib.rcParams['legend.fontsize'] = base_fs

In [ ]:
# Load the dataframes.
design_valid_plot_df = pd.read_csv("/home/akubaney/projects/na_mpnn/evaluation/evaluation_summaries/design_valid_plot.csv")
specificity_valid_plot_df = pd.read_csv("/home/akubaney/projects/na_mpnn/evaluation/evaluation_summaries/specificity_valid_plot.csv")
specificity_valid_hypersweep_plot_df = pd.read_csv("/home/akubaney/projects/na_mpnn/evaluation/evaluation_summaries/specificity_valid_hypersweep_plot.csv")
design_test_plot_df = pd.read_csv("/home/akubaney/projects/na_mpnn/evaluation/evaluation_summaries/design_test_plot.csv")
specificity_test_plot_df = pd.read_csv("/home/akubaney/projects/na_mpnn/evaluation/evaluation_summaries/specificity_test_plot.csv")

In [ ]:
plot_aggregate_metric_vs_model_step(
    design_valid_plot_df,
    metric = "Sequence recovery",
    agg = "median",
    groups = ["DNA", "DNA (protein context)", "RNA", "RNA (protein context)"],
    style = {
        **metric_vs_model_step_style,
        "show_title": False,
        "title": "Design Validation Set: Sequence Recovery vs. Model Step",
        "palette": polymer_type_palette,
        "save_name": "/home/akubaney/projects/na_mpnn/figures/matplotlib/design_valid_sequence_recovery_vs_model_step.svg",
        "legend_ncol": 2,
        "legend_figsize": (53 / 25.4, 5 / 25.4)
    },
)

In [ ]:
plot_aggregate_metric_vs_model_step(
    specificity_valid_plot_df,
    metric = "Cross-entropy",
    agg = "median",
    groups = ["Crystal", "Distillation"],
    style = {
        **metric_vs_model_step_style,
        "show_title": False,
        "title": "Specificity Validation Set: Cross Entropy vs. Model Step",
        "palette": ppm_type_palette,
        "save_name": "/home/akubaney/projects/na_mpnn/figures/matplotlib/specificity_valid_cross_entropy_vs_model_step.svg",
        "legend_figsize": (16 / 25.4, 8 / 25.4)
    },
)

In [ ]:
plot_aggregate_metric_vs_model_step(
    specificity_valid_plot_df,
    metric = "Mean absolute error",
    agg = "median",
    groups = ["Crystal", "Distillation"],
    style = {
        **metric_vs_model_step_style,
        "show_title": False,
        "title": "Specificity Validation Set: Mean Absolute Error vs. Model Step",
        "palette": ppm_type_palette,
        "save_name": "/home/akubaney/projects/na_mpnn/figures/matplotlib/specificity_valid_mean_absolute_error_vs_model_step.svg",
        "legend_figsize": (16 / 25.4, 8 / 25.4)
    },
)

In [ ]:
plot_metric_heatmap(
    specificity_valid_hypersweep_plot_df,
    metric = "Cross-entropy",
    agg = "median",
    groups = ["Distillation"],
    style = {
        **metric_heatmap_style,
        "show_title": False,
        "title": "Median Cross-Entropy",
        "cbar_label": "Median cross-entropy",
        "save_name": "/home/akubaney/projects/na_mpnn/figures/matplotlib/specificity_valid_hypersweep_distillation_cross_entropy_heatmap.svg",
    },
)

In [ ]:
plot_metric_heatmap(
    specificity_valid_hypersweep_plot_df,
    metric = "Mean absolute error",
    agg = "median",
    groups = ["Distillation"],
    style = {
        **metric_heatmap_style,
        "show_title": False,
        "title": "Median Mean Absolute Error",
        "cbar_label": "Median mean absolute error",
        "save_name": "/home/akubaney/projects/na_mpnn/figures/matplotlib/specificity_valid_hypersweep_distillation_mean_absolute_error_heatmap.svg",
    },
)

In [ ]:
plot_metric_heatmap(
    specificity_valid_hypersweep_plot_df,
    metric = "Cross-entropy",
    agg = "median",
    groups = ["Crystal"],
    style = {
        **metric_heatmap_style,
        "show_title": False,
        "title": "Median Cross-Entropy",
        "cbar_label": "Median cross-entropy",
        "save_name": "/home/akubaney/projects/na_mpnn/figures/matplotlib/specificity_valid_hypersweep_crystal_cross_entropy_heatmap.svg",
    },
)

In [ ]:
plot_metric_heatmap(
    specificity_valid_hypersweep_plot_df,
    metric = "Mean absolute error",
    agg = "median",
    groups = ["Crystal"],
    style = {
        **metric_heatmap_style,
        "show_title": False,
        "title": "Median Mean Absolute Error",
        "cbar_label": "Median mean absolute error",
        "save_name": "/home/akubaney/projects/na_mpnn/figures/matplotlib/specificity_valid_hypersweep_crystal_mean_absolute_error_heatmap.svg",
    },
)

In [ ]:
# Setup the design test without protein dataframe.
design_test_without_protein_df = design_test_plot_df[
    design_test_plot_df["protein_context"] == "without protein"
].copy()

def _assign_without_protein_group(row):
    if row.get("is_pseudoknot") is True:
        return "RNA pseudoknot"
    if row["na_complex_type"] == "ssRNA":
        return "other ssRNA"
    if row["na_complex_type"] == "ssDNA":
        return "ssDNA"
    return "other NA"

design_test_without_protein_df["Group"] = design_test_without_protein_df.apply(
    _assign_without_protein_group, axis=1
)

In [ ]:
for metric, save_stub in [
    ("Sequence recovery",        "design_test_without_protein_sequence_recovery"),
    ("GC content",               "design_test_without_protein_gc_content"),
    ("ΔGC content",              "design_test_without_protein_delta_gc_content"),
    ("AF3 pLDDT",                "design_test_without_protein_af3_plddt"),
    ("AF3 pTM",                  "design_test_without_protein_af3_ptm"),
    ("AF3 PAE",                  "design_test_without_protein_af3_pae"),
    ("AF3 C1' RMSD",             "design_test_without_protein_af3_c1_prime_rmsd"),
    ("AF3 C1' LDDT",             "design_test_without_protein_af3_c1_prime_lddt"),
    ("AF3 C1' GDDT",             "design_test_without_protein_af3_c1_prime_gddt"),
    ("RibonanzaNet pair F1",     "design_test_without_protein_ribonanza_net_pair_f1"),
    ("RibonanzaNet loop F1",     "design_test_without_protein_ribonanza_net_loop_f1"),
    ("Mean RibonanzaNet F1",     "design_test_without_protein_mean_ribonanza_net_f1"),
    ("RibonanzaNet OKS",         "design_test_without_protein_ribonanza_net_oks")
]:
    figsize = (85 / 25.4, 40 / 25.4)
    if metric in (
        "RibonanzaNet OKS",
    ):
        protein_groups = ["RNA pseudoknot"]
        layout_groups = ["RNA pseudoknot", "other ssRNA", "ssDNA", "other NA"]
        layout_group_widths = {
            "RNA pseudoknot": 4,
            "other ssRNA": 4,
            "ssDNA": 1,
            "other NA": 1,
        }
    elif metric in (
        "RibonanzaNet pair F1",
        "RibonanzaNet loop F1",
        "Mean RibonanzaNet F1",
    ):
        protein_groups = ["RNA pseudoknot", "other ssRNA"]
        layout_groups = ["RNA pseudoknot", "other ssRNA", "ssDNA", "other NA"]
        layout_group_widths = {
            "RNA pseudoknot": 4,
            "other ssRNA": 4,
            "ssDNA": 1,
            "other NA": 1,
        }
    else:
        protein_groups = ["RNA pseudoknot", "other ssRNA", "ssDNA", "other NA"]
        layout_groups = None
        layout_group_widths = None

    if metric in (
        "GC content",
        "ΔGC content",
        "RibonanzaNet pair F1",
        "RibonanzaNet loop F1",
        "Mean RibonanzaNet F1",
        "AF3 C1' LDDT",
        "AF3 C1' GDDT",
        "AF3 pLDDT",
        "AF3 pTM",
        "AF3 PAE",
    ):
        figsize = (60 / 25.4, 40 / 25.4)
        legend_ncol = 1
        legend_figsize = (15 / 25.4, 15 / 25.4)
        legend_handlelength = box_style["legend_handlelength"]
    else:
        figsize = box_style["figsize"]
        legend_ncol = 4
        legend_figsize = (80 / 25.4, 4 / 25.4)
        legend_handlelength = 3

    plot_box(
        design_test_without_protein_df,
        metric=metric,
        groups=protein_groups,
        models=design_models_without_protein,
        style={
            **box_style,
            "assert_complete_group_coverage": True,
            "coverage_models": design_models_without_protein,
            "show_legend": True,
            "plot_by": "Model",
            "group_by": "Group",
            "layout_groups": layout_groups,
            "layout_group_widths": layout_group_widths,
            "palette": design_model_palette,
            "model_name_to_label": model_name_to_label,
            "ymin": -0.05 if metric == "Sequence recovery" else None,
            "ytick_range": (0, 1) if metric == "Sequence recovery" else None,
            "save_name": f"/home/akubaney/projects/na_mpnn/figures/matplotlib/{save_stub}.svg",
            "legend_ncol": legend_ncol,
            "legend_figsize": legend_figsize,
            "figsize": figsize,
            "legend_handlelength": legend_handlelength,
        },
    )

In [ ]:
design_test_with_protein_df = design_test_plot_df[
    design_test_plot_df["protein_context"] == "with protein"
].copy()

def _assign_with_protein_group(row):
    if row["na_complex_type"] == "ssRNA":
        return "protein-ssRNA"
    return "protein-other NA"

design_test_with_protein_df["Group"] = design_test_with_protein_df.apply(
    _assign_with_protein_group, axis=1
)

# Rename "AF3 iPTM" column to "AF3 ipTM" for better readability in the plot.
design_test_with_protein_df = design_test_with_protein_df.rename(columns={"AF3 iPTM": "AF3 ipTM"})

In [ ]:
for metric, save_stub in [
    ("Sequence recovery",             "design_test_with_protein_sequence_recovery"),
    ("GC content",                    "design_test_with_protein_gc_content"),
    ("ΔGC content",                   "design_test_with_protein_delta_gc_content"),
    ("AF3 ipTM",                      "design_test_with_protein_af3_iptm"),
    ("AF3 protein-aligned C1' RMSD",  "design_test_with_protein_af3_protein_aligned_na_c1_prime_rmsd"),
]:
    protein_groups = ["protein-ssRNA", "protein-other NA"]
    layout_groups = ["protein-ssRNA", "protein-other NA", "blank1", "blank2"]
    layout_group_widths = {
        "protein-ssRNA": 5,
        "protein-other NA": 2,
        "blank1": 2,
        "blank2": 1,
    }

    if metric == "AF3 protein-aligned C1' RMSD":
        hline_value = 5
    elif metric == "AF3 ipTM":
        hline_value = 0.7
    else:
        hline_value = None
    
    if metric in (
        "Sequence recovery",
        "GC content",
        "ΔGC content",
    ):
        figsize = (60 / 25.4, 40 / 25.4)
    else:
        figsize = box_style["figsize"]

    plot_box(
        design_test_with_protein_df,
        metric=metric,
        groups=protein_groups,
        models=design_models_with_protein,
        style={
            **box_style,
            "assert_complete_group_coverage": True,
            "coverage_models": design_models_with_protein,
            "show_legend": True,
            "plot_by": "Model",
            "group_by": "Group",
            "palette": design_model_palette,
            "model_name_to_label": model_name_to_label,
            "layout_groups": layout_groups,
            "layout_group_widths": layout_group_widths,
            "ymin": -0.05 if metric == "Sequence recovery" else None,
            "ytick_range": (0, 1) if metric == "Sequence recovery" else None,
            "save_name": f"/home/akubaney/projects/na_mpnn/figures/matplotlib/{save_stub}.svg",
            "legend_figsize": (25 / 25.4, 14 / 25.4),
            "figsize": figsize,
            "hline_value": hline_value,
        },
    )

In [ ]:
print(                                                                                                                                                                
      design_test_with_protein_df.assign(                                                                                                                             
          passes=(design_test_with_protein_df["AF3 protein-aligned C1' RMSD"] < 5)                                                                                      
          & (design_test_with_protein_df["AF3 ipTM"] > 0.7)
      )                                                                                                                                                                 
      .groupby(["Model", "Group"])["passes"]
      .mean()                                                                                                                                                           
      .unstack("Group")
  )  

In [ ]:
# Design test example metrics
example_metrics = [
    "Model",
    "Name",
    "Sequence recovery",
    "AF3 C1' RMSD",
    "RibonanzaNet OKS",
    "AF3 protein-aligned C1' RMSD",
    "AF3 iPTM"
]
example_names = [
    "4plx_2",
    "4l81_14",
    "6ftu_12",
    "3dh3_4",
    "6u82_21",
    "2np2_4",
    "2m8k_21",
    "7k9e_24"
]
# Fix 
design_test_plot_df[(design_test_plot_df.Model == "na_mpnn") & (design_test_plot_df.Name.isin(example_names))][example_metrics]

In [ ]:
specificity_plot_models = ["na_mpnn", "deeppbs", "rclamps"]
specificity_coverage_models = ["na_mpnn", "deeppbs"]

# Grab the distillation subset and set group to the family.
specificity_distillation_family_df = specificity_test_plot_df[
    specificity_test_plot_df["dataset_name"] == "rf2na_distillation_cis_bp"
].copy()
specificity_distillation_family_df["Group"] = (
    specificity_distillation_family_df["family"]
)

# Subset to examples that DeepPBS was able to run on.
distillation_shared_ids = set(
    specificity_distillation_family_df.loc[
        specificity_distillation_family_df["Model"] == "na_mpnn", "id"
    ]
) & set(
    specificity_distillation_family_df.loc[
        specificity_distillation_family_df["Model"] == "deeppbs", "id"
    ]
)
specificity_distillation_family_df = specificity_distillation_family_df[
    specificity_distillation_family_df["id"].isin(distillation_shared_ids)
]

# Sort the families by counts, except for other.
distillation_family_counts = (
    specificity_distillation_family_df.groupby("family").size().sort_values(ascending=False)
)
distillation_family_order = [f for f in distillation_family_counts.index if f != "Other"]
if "Other" in distillation_family_counts.index:
    distillation_family_order.append("Other")

In [ ]:
# Make the plots.
for metric, save_suffix in [
    ("Cross-entropy", "cross_entropy_by_family"),
    ("Mean absolute error", "mean_absolute_error_by_family"),
]:
    plot_box(
        specificity_distillation_family_df,
        metric=metric,
        groups=distillation_family_order,
        models=specificity_plot_models,
        style={
            **box_style,
            "plot_kind": "box",
            "assert_complete_group_coverage": True,
            "coverage_models": specificity_coverage_models,
            "show_legend": True,
            "plot_by": "Model",
            "group_by": "Group",
            "palette": specificity_model_palette,
            "model_name_to_label": model_name_to_label,
            "x_label": "Protein family",
            "x_tick_rotation": 30,
            "x_tick_horizontalalignment": "right",
            "save_name": (
                "/home/akubaney/projects/na_mpnn/figures/matplotlib/"
                f"specificity_test_distillation_{save_suffix}.svg"
            ),
            "figsize": (80 / 25.4, 40 / 25.4),
            "legend_ncol": 3,
            "legend_figsize": (45 / 25.4, 4.5 / 25.4),
            "subplot_wspace": 0,
            "constrained_w_pad": 0,
            "constrained_wspace": 0,
            "constrained_layout_rect": (0, 0.22, 1, 0.78),
            "x_tick_rotation_mode": "anchor",
            "x_axis_in_layout": False,
        },
    )

In [ ]:
# Grab the crystal subset and set group to the family.
specificity_crystal_family_df = specificity_test_plot_df[
    specificity_test_plot_df["dataset_name"] == "rcsb_cif_na"
].copy()
specificity_crystal_family_df["Group"] = (
    specificity_crystal_family_df["family"]
)

# Subset to examples that DeepPBS was able to run on.
crystal_shared_ids = set(
    specificity_crystal_family_df.loc[
        specificity_crystal_family_df["Model"] == "na_mpnn", "id"
    ]
) & set(
    specificity_crystal_family_df.loc[
        specificity_crystal_family_df["Model"] == "deeppbs", "id"
    ]
)
specificity_crystal_family_df = specificity_crystal_family_df[
    specificity_crystal_family_df["id"].isin(crystal_shared_ids)
]

# Move singleton families into "Other" before plotting.
crystal_family_id_counts = specificity_crystal_family_df.groupby("family")["id"].nunique()
crystal_singleton_families = crystal_family_id_counts[
    (crystal_family_id_counts == 1) & (crystal_family_id_counts.index != "Other")
].index

specificity_crystal_family_df.loc[
    specificity_crystal_family_df["family"].isin(crystal_singleton_families),
    "family",
] = "Other"
specificity_crystal_family_df["Group"] = specificity_crystal_family_df["family"]

# Sort the families by counts, except for other.
crystal_family_counts = (
    specificity_crystal_family_df.groupby("family").size().sort_values(ascending=False)
)
crystal_family_order = [f for f in crystal_family_counts.index if f != "Other"]
if "Other" in crystal_family_counts.index:
    crystal_family_order.append("Other")

In [ ]:
# Make the plots.
for metric, save_suffix in [
    ("Cross-entropy", "cross_entropy_by_family"),
    ("Mean absolute error", "mean_absolute_error_by_family"),
]:
    plot_box(
        specificity_crystal_family_df,
        metric=metric,
        groups=crystal_family_order,
        models=specificity_plot_models,
        style={
            **box_style,
            "plot_kind": "box",
            "assert_complete_group_coverage": True,
            "coverage_models": specificity_coverage_models,
            "show_legend": True,
            "plot_by": "Model",
            "group_by": "Group",
            "palette": specificity_model_palette,
            "model_name_to_label": model_name_to_label,
            "x_label": "Protein family",
            "x_tick_rotation": 30,
            "x_tick_horizontalalignment": "right",
            "save_name": (
                "/home/akubaney/projects/na_mpnn/figures/matplotlib/"
                f"specificity_test_crystal_{save_suffix}.svg"
            ),
            "figsize": (67 / 25.4, 40 / 25.4),
            "legend_ncol": 3,
            "legend_figsize": (45 / 25.4, 4.5 / 25.4),
            "subplot_wspace": 0,
            "constrained_w_pad": 0,
            "constrained_wspace": 0,
            "constrained_layout_rect": (0, 0.30, 1, 0.70),
            "x_tick_rotation_mode": "anchor",
            "x_axis_in_layout": False,
        },
    )

In [ ]:
entries_and_deltas = []
for structure_path in specificity_test_plot_df["structure_path"].unique():
    structure_df = specificity_test_plot_df[specificity_test_plot_df["structure_path"] == structure_path]

    if "na_mpnn" not in structure_df["Model"].values or "deeppbs" not in structure_df["Model"].values:
        continue

    na_mpnn_mae = structure_df[structure_df["Model"] == "na_mpnn"]["Mean absolute error"].values[0]
    deeppbs_mae = structure_df[structure_df["Model"] == "deeppbs"]["Mean absolute error"].values[0]
    mae_delta = na_mpnn_mae - deeppbs_mae

    na_mpnn_cross_entropy = structure_df[structure_df["Model"] == "na_mpnn"]["Cross-entropy"].values[0]
    deeppbs_cross_entropy = structure_df[structure_df["Model"] == "deeppbs"]["Cross-entropy"].values[0]
    cross_entropy_delta = na_mpnn_cross_entropy - deeppbs_cross_entropy

    entries_and_deltas.append((
        structure_path,
        mae_delta,
        na_mpnn_mae,
        cross_entropy_delta,
        na_mpnn_cross_entropy
    ))

entries_and_deltas = sorted(entries_and_deltas, key=lambda x: x[2])
for i in range(0, len(entries_and_deltas)):
    print(entries_and_deltas[i])